In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("librerias importadas")

librerias importadas


In [ ]:
# creacion de carpetas si no existen
BASE_DIR = Path("/content/")
DATA_PROCESSED = BASE_DIR/"datos"/"procesados"
DOCUMENTOS = BASE_DIR/"documentos"

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
DOCUMENTOS.mkdir(parents=True, exist_ok=True)

print(f"Datos procesados: {DATA_PROCESSED}")
print(f"Documentos: {DOCUMENTOS}")

Datos procesados: /content/datos/procesados
Documentos: /content/documentos


In [ ]:
#PUNTO 1 cargar dataset integrado de la GUIA01

ruta_entrada = DATA_PROCESSED/"dataset_integrado.csv"
df = pd.read_csv(ruta_entrada)
#dataset cargado
print(f"Dataset cargado: {ruta_entrada}")
print(f"Dimensiones: {df.shape[0]} filas, {df.shape[1]} columnas")
df.head()

Dataset cargado: /content/datos/procesados/dataset_integrado.csv
Dimensiones: 20144 filas, 19 columnas


,company_id,product_area,event_date,ticket_count,deployment_count,failed_deployments,avg_lead_time_hours,rollback_count,incident_count,avg_resolution_time_hours,total_downtime_min,avg_cpu_usage_pct,avg_memory_usage_pct,avg_response_time_ms,avg_error_rate_pct,avg_availability_pct,avg_requests_per_minute,deployment_failure_rate_pct,rollback_rate_pct
0,100001,analytics,2026-01-01,3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,56.51,31.39,265.0,7.24,98.422,7051.0,NaN,NaN
1,100001,analytics,2026-01-02,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,100001,analytics,2026-01-03,2,1.0,0.0,18.29,0.0,NaN,NaN,NaN,55.67,53.48,736.0,3.00,96.012,4186.0,0.0,0.0
3,100001,analytics,2026-01-04,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,65.00,75.00,313.0,5.59,95.885,854.0,NaN,NaN
4,100001,analytics,2026-01-05,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [ ]:
# CREAR COPIAS DE TRABAJO / buena practica.
# df_original: se conserva intacta como respaldo, nunca se modifica
# df_limpio: sobre esta se aplican todos los cambios de limpieza

df_original = df.copy()
df_limpio = df.copy()

print("Copias de trabajo creadas ---------")
print(f"Original: {df_original.shape}")
print(f"Para limpiar: {df_limpio.shape}")

Copias de trabajo creadas ---------
Original: (20144, 19)
Para limpiar: (20144, 19)


In [ ]:
# PUNTO 2: DIAGNOSTICO INICIAL DE CALIDAD
# Revisamos dimensiones y nombres de columnas antes de tocar nada

print("Dimensiones: -----------------")
print(f"Filas: {df_limpio.shape[0]}, Columnas: {df_limpio.shape[1]}")

print("\nNombres de las columnas: ----------")
print(df_limpio.columns.tolist())


#NOTA/recordatorio
#deployment_failure_rate_pct y rollback_rate_pct ya vienen calculadas desde la Guía 1,

Dimensiones: -----------------
Filas: 20144, Columnas: 19

Nombres de las columnas: ----------
['company_id', 'product_area', 'event_date', 'ticket_count', 'deployment_count', 'failed_deployments', 'avg_lead_time_hours', 'rollback_count', 'incident_count', 'avg_resolution_time_hours', 'total_downtime_min', 'avg_cpu_usage_pct', 'avg_memory_usage_pct', 'avg_response_time_ms', 'avg_error_rate_pct', 'avg_availability_pct', 'avg_requests_per_minute', 'deployment_failure_rate_pct', 'rollback_rate_pct']


In [ ]:
# TIPOS DE DATOS
# Verificamos que cada columna tenga el tipo correcto

# (fechas como texto, numeros como object etc.)

# REMINDER: pendientes de corregir mas adelante
# - event_date esta como object, hay que convertirla a datetime
# deployment_count, failed_deployments, incident_count, rollback_count,
# avg_resolution_time_hours, total_downtime_min estan como float64
# NaN convertiran a int despues de rellenar con 0
print("Tipos de datos ========================")
print(df_limpio.dtypes)

Tipos de datos ========================
company_id                       int64
product_area                    object
event_date                      object
ticket_count                     int64
deployment_count               float64
failed_deployments             float64
avg_lead_time_hours            float64
rollback_count                 float64
incident_count                 float64
avg_resolution_time_hours      float64
total_downtime_min             float64
avg_cpu_usage_pct              float64
avg_memory_usage_pct           float64
avg_response_time_ms           float64
avg_error_rate_pct             float64
avg_availability_pct           float64
avg_requests_per_minute        float64
deployment_failure_rate_pct    float64
rollback_rate_pct              float64
dtype: object


In [ ]:
# PUNTO 3: IDENTIFICAR COLUMNAS CON VALORES NULOS
# Nulos esperados: vienen del merge outer, no significan error de captura

print("Valores nulos ===========")
nulos = df_limpio.isnull().sum()
print(nulos)

print("\nPorcentaje de nulos ====================")
porcentaje_nulos = (df_limpio.isnull().mean() * 100).round(2)
print(porcentaje_nulos)

# Encontramos que los nulos NO son error de captura, son ausencia real de eventos.
# deployment_count  47.59 (48&) de dias sin evento (normal, no despliegan a diario)
# - incident_count 81.94 (82%) de dias sin incidentes (esperado, son eventos raros)

# Mas adelante realizamos lo siguiente:
#Conteos se llenan con 0, (donde no hubo event)
# Metrica de performance se dejan como NaN
# Se marcaran con una caracteristica para no crear datos de monitoreo existentes

Valores nulos ===========
company_id                         0
product_area                       0
event_date                         0
ticket_count                       0
deployment_count                9586
failed_deployments              9586
avg_lead_time_hours             9586
rollback_count                  9586
incident_count                 16506
avg_resolution_time_hours      16506
total_downtime_min             16506
avg_cpu_usage_pct               9648
avg_memory_usage_pct            9648
avg_response_time_ms            9648
avg_error_rate_pct              9648
avg_availability_pct            9648
avg_requests_per_minute         9648
deployment_failure_rate_pct     9586
rollback_rate_pct               9586
dtype: int64

Porcentaje de nulos ====================
company_id                      0.00
product_area                    0.00
event_date                      0.00
ticket_count                    0.00
deployment_count               47.59
failed_deployments             

In [ ]:
# PUNTO 4: IDENTIFICAR REGISTROS DUPLICADOS

print("Registros duplicados -----------")
duplicados = df_limpio.duplicated().sum()
print(f"Registros duplicados: {duplicados}")

# Hallazgo: 0 duplicados exactos. El merge outer de la guia 1 no genero
# filas repetidas, por lo que no se requiere drop_duplicates() en este paso.

Registros duplicados -----------
Registros duplicados: 0


In [ ]:
# VALORES UNICOS EN PRODUCT_AREA
# Revisamos si hay inconsistencias de texto antes de normalizar

print("Valores unicos en product_area:")
print(df_limpio["product_area"].unique())

Valores unicos en product_area:
['analytics' 'auth' 'billing' 'mobile' 'notifications' 'data_pipeline']


In [ ]:
# Valores unicos reales del proyecto: ya vienen en minuscula y sin
# inconsistencias de mayusculas/espacios visibles, pero se normalizara
# el formato (Title Case) para presentacion consistente mas adelante.

mapeo_areas = {
    "analytics": "Analytics",
    "auth": "Auth",
    "billing": "Billing",
    "mobile": "Mobile",
    "notifications": "Notifications",
    "data_pipeline": "Data Pipeline"
}

In [ ]:
# INFORMACION GENERAL - resumen completo de tipos y no-nulls

df_limpio.info()

# Cierre del diagnostico inicial:
# - 20.144 filas, 19 columnas, 0 duplicados
# - 6 valores limpios (analytics, auth, billing, mobile,
#   notifications, data_pipeline), sin inconsistencias de texto
# - event_date como object, pendiente convertir a datetime
# - Nulos concentrados en deployments (~48%), incidents (~82%) y
#   performance (~48%), todos por ausencia real de eventos, no error

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 20144 entries, 0 to 20143
Data columns (total 19 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   company_id                   20144 non-null  int64  
 1   product_area                 20144 non-null  object 
 2   event_date                   20144 non-null  object 
 3   ticket_count                 20144 non-null  int64  
 4   deployment_count             10558 non-null  float64
 5   failed_deployments           10558 non-null  float64
 6   avg_lead_time_hours          10558 non-null  float64
 7   rollback_count               10558 non-null  float64
 8   incident_count               3638 non-null   float64
 9   avg_resolution_time_hours    3638 non-null   float64
 10  total_downtime_min           3638 non-null   float64
 11  avg_cpu_usage_pct            10496 non-null  float64
 12  avg_memory_usage_pct         10496 non-null  float64
 13  avg_response_tim

In [ ]:
# PUNTO 6: NORMALIZAR NOMBRES DE COLUMNAS
# Ya vienen en snake_case, pero aplicamos el proceso como buena practica
# defensiva por si hubiera espacios o mayusculas ocultas

print("ANTES:", df_limpio.columns.tolist())

df_limpio.columns = (
    df_limpio.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_", regex=False)
)

print("\nDESPUES:", df_limpio.columns.tolist())

ANTES: ['company_id', 'product_area', 'event_date', 'ticket_count', 'deployment_count', 'failed_deployments', 'avg_lead_time_hours', 'rollback_count', 'incident_count', 'avg_resolution_time_hours', 'total_downtime_min', 'avg_cpu_usage_pct', 'avg_memory_usage_pct', 'avg_response_time_ms', 'avg_error_rate_pct', 'avg_availability_pct', 'avg_requests_per_minute', 'deployment_failure_rate_pct', 'rollback_rate_pct']

DESPUES: ['company_id', 'product_area', 'event_date', 'ticket_count', 'deployment_count', 'failed_deployments', 'avg_lead_time_hours', 'rollback_count', 'incident_count', 'avg_resolution_time_hours', 'total_downtime_min', 'avg_cpu_usage_pct', 'avg_memory_usage_pct', 'avg_response_time_ms', 'avg_error_rate_pct', 'avg_availability_pct', 'avg_requests_per_minute', 'deployment_failure_rate_pct', 'rollback_rate_pct']


In [ ]:
# PUNTO 7 (parte 1): ELIMINAR ESPACIOS EN TEXTOS

columnas_texto = df_limpio.select_dtypes(include="object").columns
print(f"Columnas de texto: {columnas_texto.tolist()}")

for columna in columnas_texto:
    df_limpio[columna] = df_limpio[columna].astype("string").str.strip()

print("Espacios eliminados de todas las columnas de texto")

Columnas de texto: ['product_area', 'event_date']
Espacios eliminados de todas las columnas de texto


In [ ]:
# PUNTO 7 (parte 2): NORMALIZAR PRODUCT_AREA
# Estandarizamos a Title Case para presentacion consistente.

print("PRODUCT_AREA - ANTES:", df_limpio["product_area"].unique().tolist())

df_limpio["product_area"] = df_limpio["product_area"].str.lower()

mapeo_areas = {
    "analytics": "Analytics",
    "auth": "Auth",
    "billing": "Billing",
    "mobile": "Mobile",
    "notifications": "Notifications",
    "data_pipeline": "Data Pipeline"
}

df_limpio["product_area"] = df_limpio["product_area"].replace(mapeo_areas)

print("PRODUCT_AREA - DESPUES:", df_limpio["product_area"].unique().tolist())

PRODUCT_AREA - ANTES: ['analytics', 'auth', 'billing', 'mobile', 'notifications', 'data_pipeline']
PRODUCT_AREA - DESPUES: ['Analytics', 'Auth', 'Billing', 'Mobile', 'Notifications', 'Data Pipeline']


In [ ]:
# PUNTO 8: CONVERTIR FECHA A DATETIME

df_limpio["event_date"] = pd.to_datetime(df_limpio["event_date"], errors="coerce")

print("event_date convertida a:", df_limpio["event_date"].dtype)
print(df_limpio[["event_date"]].head())

event_date convertida a: datetime64[ns]
  event_date
0 2026-01-01
1 2026-01-02
2 2026-01-03
3 2026-01-04
4 2026-01-05


In [ ]:
# RELLENAR CONTEOS CON 0
# Estas columnas son NaN porque ese dia no hubo deployments/incidentes,
# no porque falte un dato real. Por eso se rellenan con 0, no se elimina la fila.

columnas_conteo = [
    "deployment_count", "failed_deployments", "rollback_count",
    "incident_count", "total_downtime_min"
]

for columna in columnas_conteo:
    df_limpio[columna] = df_limpio[columna].fillna(0)

print("Conteos rellenados con 0")
print(df_limpio[columnas_conteo].isnull().sum())

Conteos rellenados con 0
deployment_count      0
failed_deployments    0
rollback_count        0
incident_count        0
total_downtime_min    0
dtype: int64


In [ ]:
# CONVERTIR CONTEOS A ENTERO
# Ahora que no tienen NaN, vuelven a int

for columna in columnas_conteo:
    df_limpio[columna] = df_limpio[columna].astype(int)

print(df_limpio[columnas_conteo].dtypes)

deployment_count      int64
failed_deployments    int64
rollback_count        int64
incident_count        int64
total_downtime_min    int64
dtype: object


In [ ]:
# PUNTO 9: CORREGIR VALORES INVALIDOS
# Revisamos valores negativos en columnas que nunca deberian tenerlos

columnas_no_negativas = [
    "avg_cpu_usage_pct", "avg_memory_usage_pct", "avg_response_time_ms",
    "avg_error_rate_pct", "avg_availability_pct", "avg_requests_per_minute",
    "avg_lead_time_hours", "avg_resolution_time_hours"
]

for columna in columnas_no_negativas:
    negativos = df_limpio[df_limpio[columna] < 0]
    print(f"{columna}: {len(negativos)} valores negativos")


# Resultado: no se encontraron valores negativos en ninguna columna de
# performance/tiempo. No requiere correccion en este paso.

avg_cpu_usage_pct: 0 valores negativos
avg_memory_usage_pct: 0 valores negativos
avg_response_time_ms: 0 valores negativos
avg_error_rate_pct: 0 valores negativos
avg_availability_pct: 0 valores negativos
avg_requests_per_minute: 0 valores negativos
avg_lead_time_hours: 0 valores negativos
avg_resolution_time_hours: 0 valores negativos


In [ ]:
# PUNTO 10: CARACTERISTICAS TEMPORALES (nuevas 1-5)


#extrae el anio 2026-01-15 = 2026
df_limpio["anio"] = df_limpio["event_date"].dt.year
#extrae el mes 2026-01-15 = 01
df_limpio["mes"] = df_limpio["event_date"].dt.month
#extrae el dia =2026-01-15 = 15
df_limpio["dia"] = df_limpio["event_date"].dt.day
#nombre del dia de semana
df_limpio["dia_semana"] = df_limpio["event_date"].dt.day_name()
#extrae el dia de la semana 2026-01-15 = 5 (0 al 6 = lunes a domingo)
df_limpio["es_fin_semana"] = df_limpio["event_date"].dt.dayofweek >= 5

print("Caracteristicas temporales creadas:")
df_limpio[["event_date", "anio", "mes", "dia", "dia_semana", "es_fin_semana"]].head()

Caracteristicas temporales creadas:


,event_date,anio,mes,dia,dia_semana,es_fin_semana
0,2026-01-01,2026,1,1,Thursday,False
1,2026-01-02,2026,1,2,Friday,False
2,2026-01-03,2026,1,3,Saturday,True
3,2026-01-04,2026,1,4,Sunday,True
4,2026-01-05,2026,1,5,Monday,False


In [ ]:
# CARACTERISTICA NUEVA 6:  dias sin monitoreo
# Marca explicitamente cuando no hubo registro de performance ese dia,
# en vez de dejar el NaN sin explicacion en el dataset final

columnas_performance = [
    "avg_cpu_usage_pct", "avg_memory_usage_pct", "avg_response_time_ms",
    "avg_error_rate_pct", "avg_availability_pct", "avg_requests_per_minute"
]
#true/false sin monitoreo
df_limpio["sin_datos_monitoreo"] = df_limpio[columnas_performance].isnull().any(axis=1)

print("Dias sin datos de monitoreo:", df_limpio["sin_datos_monitoreo"].sum())
#/ 9648 dias setted como True

Dias sin datos de monitoreo: 9648


In [ ]:
# PUNTO 11: REPORTE DE CALIDAD
# Tabla resumen de cambios realizados:
# tipo de dato, nulos y valores unicos por columna

reporte_calidad = pd.DataFrame({
    "columna": df_limpio.columns,
    "tipo_dato": [str(df_limpio[col].dtype) for col in df_limpio.columns],
    "valores_nulos": [df_limpio[col].isnull().sum() for col in df_limpio.columns],
    "valores_unicos": [df_limpio[col].nunique(dropna=False) for col in df_limpio.columns]
})

print("REPORTE DE CALIDAD:")
reporte_calidad

REPORTE DE CALIDAD:


,columna,tipo_dato,valores_nulos,valores_unicos
0,company_id,int64,0,25
1,product_area,string,0,6
2,event_date,datetime64[ns],0,181
3,ticket_count,int64,0,12
4,deployment_count,int64,0,8
5,failed_deployments,int64,0,4
6,avg_lead_time_hours,float64,9586,3845
7,rollback_count,int64,0,4
8,incident_count,int64,0,5
9,avg_resolution_time_hours,float64,16506,1270


In [ ]:
# GUARDAR REPORTE DE CALIDAD EN EXCEL

ruta_reporte = DOCUMENTOS/"reporte_calidad_ProyctoFinal02_equipo.xlsx"
reporte_calidad.to_excel(ruta_reporte, index=False)
print(f"Reporte guardado: {ruta_reporte}")

Reporte guardado: /content/documentos/reporte_calidad_ProyctoFinal02_equipo.xlsx


In [ ]:
# PUNTO 12: GUARDAR DATASET LIMPIO DEL PROYECTO

ruta_salida = DATA_PROCESSED/"dataset_limpio.csv"
df_limpio.to_csv(ruta_salida, index=False, encoding="utf-8")
print(f"Dataset limpio guardado: {ruta_salida}")

Dataset limpio guardado: /content/datos/procesados/dataset_limpio.csv
